# 05 Train and Tune the RE Transformer

This notebook now supports named, controlled relation extraction experiments. The
report split and random seed remain fixed. Change only `RE_EXPERIMENT_NAME` to run:

- BERT with more negatives and no class weighting;
- PubMedBERT with the same settings;
- generic versus type-aware entity markers;
- capped class-weighted loss.

Validation probabilities are used to tune a positive-relation threshold. The selected
threshold is then frozen before test evaluation. Each experiment writes to a separate
model, result, and prediction path, so the original BERT baseline is retained.


In [ ]:
from __future__ import annotations

NOTEBOOK_VERSION = "2026-07-25-pair-compaction-v2"

import inspect
import json
import os
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from dotenv import load_dotenv
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing pyproject.toml")


PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / ".env")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

HF_HOME = os.environ.get("HF_HOME")
if HF_HOME:
    os.environ.setdefault("HF_HOME", HF_HOME)

RUN_NAME = os.getenv("RADGRAPH_XL_RUN_NAME", "full_2300")
RUN_ROOT = PROJECT_ROOT / "outputs" / RUN_NAME
INTERIM_DIR = RUN_ROOT / "interim"
RESULTS_DIR = RUN_ROOT / "results"
PREDICTIONS_DIR = RUN_ROOT / "predictions"

NER_JSONL = INTERIM_DIR / "ner_dataset.jsonl"
RE_CANDIDATES_CSV = INTERIM_DIR / "re_candidate_pool.csv"
LABEL_MAPS_JSON = INTERIM_DIR / "label_maps.json"
PREP_SUMMARY_JSON = INTERIM_DIR / "prep_summary.json"

assert NER_JSONL.exists(), "Run notebook 03 before notebook 05."
assert RE_CANDIDATES_CSV.exists(), "Missing re_candidate_pool.csv. Run notebook 03."
assert LABEL_MAPS_JSON.exists(), "Missing label_maps.json. Run notebook 03."
assert PREP_SUMMARY_JSON.exists(), "Missing prep_summary.json. Run notebook 03."

# Every profile keeps RANDOM_SEED=42 and the same report split. The profiles are
# arranged so that each adjacent comparison changes one methodological factor.
RE_EXPERIMENTS = {
    "bert_base_uncased": {
        "checkpoint": "bert-base-uncased",
        "marker_design": "generic",
        "negative_ratio": 3,
        "weighted_loss": True,
        "class_weight_cap": None,
    },
    "bert_generic_neg5_unweighted": {
        "checkpoint": "bert-base-uncased",
        "marker_design": "generic",
        "negative_ratio": 5,
        "weighted_loss": False,
        "class_weight_cap": None,
    },
    "pubmedbert_generic_neg5_unweighted": {
        "checkpoint": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
        "marker_design": "generic",
        "negative_ratio": 5,
        "weighted_loss": False,
        "class_weight_cap": None,
    },
    "pubmedbert_typed_neg5_unweighted": {
        "checkpoint": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
        "marker_design": "type_aware",
        "negative_ratio": 5,
        "weighted_loss": False,
        "class_weight_cap": None,
    },
    "pubmedbert_typed_neg5_capped": {
        "checkpoint": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
        "marker_design": "type_aware",
        "negative_ratio": 5,
        "weighted_loss": True,
        "class_weight_cap": 5.0,
    },
}

# Recommended first new run. It tests whether more negative examples and removing
# inverse-frequency class weights improve the baseline's low precision.
RE_EXPERIMENT_NAME = "bert_generic_neg5_unweighted"

assert RE_EXPERIMENT_NAME in RE_EXPERIMENTS
experiment = RE_EXPERIMENTS[RE_EXPERIMENT_NAME]
MODEL_CHECKPOINT = experiment["checkpoint"]
MARKER_DESIGN = experiment["marker_design"]
TRAIN_NEGATIVE_TO_POSITIVE_RATIO = experiment["negative_ratio"]
USE_CLASS_WEIGHTED_LOSS = experiment["weighted_loss"]
CLASS_WEIGHT_CAP = experiment["class_weight_cap"]

MODEL_DIR = RUN_ROOT / "models" / f"re_{RE_EXPERIMENT_NAME}"
for path in [RESULTS_DIR, PREDICTIONS_DIR, MODEL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

MAX_LENGTH = 256
CONTEXT_WINDOW = 64
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
WEIGHT_DECAY = 0.01
LR_SCHEDULER_TYPE = "linear"
WARMUP_RATIO = 0.0
GRADIENT_ACCUMULATION_STEPS = 1
MAX_GRAD_NORM = 1.0

# The seed is deliberately unchanged across every experiment.
RANDOM_SEED = 42

# Validation and test retain every negative pair for realistic precision estimates.
EVAL_NEGATIVE_TO_POSITIVE_RATIO = None
CANDIDATE_CHUNK_SIZE = 100_000

# Tune this grid on validation predictions only. The chosen threshold is applied once
# to the test set and saved with the run configuration.
POSITIVE_THRESHOLD_GRID = np.round(np.arange(0.20, 0.81, 0.05), 2).tolist()

MAX_TRAIN_SAMPLES = None
MAX_EVAL_SAMPLES = None
MAX_TEST_SAMPLES = None

GENERIC_SPECIAL_TOKENS = ["[HEAD]", "[/HEAD]", "[TAIL]", "[/TAIL]"]
TYPE_AWARE_SPECIAL_TOKENS = [
    f"[{slash}{role}_{entity_type}]"
    for role in ["HEAD", "TAIL"]
    for entity_type in ["ANAT", "OBS"]
    for slash in ["", "/"]
]
SPECIAL_TOKENS = (
    GENERIC_SPECIAL_TOKENS
    if MARKER_DESIGN == "generic"
    else TYPE_AWARE_SPECIAL_TOKENS
)

set_seed(RANDOM_SEED)

print("Notebook version:", NOTEBOOK_VERSION)

print(
    {
        "project_root": str(PROJECT_ROOT),
        "run_name": RUN_NAME,
        "re_experiment": RE_EXPERIMENT_NAME,
        "model_checkpoint": MODEL_CHECKPOINT,
        "marker_design": MARKER_DESIGN,
        "negative_ratio": TRAIN_NEGATIVE_TO_POSITIVE_RATIO,
        "weighted_loss": USE_CLASS_WEIGHTED_LOSS,
        "class_weight_cap": CLASS_WEIGHT_CAP,
        "random_seed": RANDOM_SEED,
        "cuda_available": torch.cuda.is_available(),
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    }
)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select the project's .venv as the notebook kernel.")


In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                rows.append(json.loads(line))
    return rows


documents = load_jsonl(NER_JSONL)
tokens_by_doc = {row["doc_id"]: row["tokens"] for row in documents}

label_maps = json.loads(LABEL_MAPS_JSON.read_text(encoding="utf-8"))
prep_summary = json.loads(PREP_SUMMARY_JSON.read_text(encoding="utf-8"))
label_to_id = {label: int(index) for label, index in label_maps["relation_label_to_id"].items()}
id_to_label = {int(index): label for index, label in label_maps["relation_id_to_label"].items()}
label_names = [id_to_label[index] for index in sorted(id_to_label)]
no_relation_id = label_to_id["no_relation"]
positive_label_ids = [index for index, label in id_to_label.items() if label != "no_relation"]

expected_columns = {
    "source",
    "doc_id",
    "dataset",
    "split",
    "head_entity_id",
    "tail_entity_id",
    "head_start",
    "head_end",
    "tail_start",
    "tail_end",
    "head_label",
    "tail_label",
    "distance",
    "is_gold",
    "label",
}
candidate_columns = list(pd.read_csv(RE_CANDIDATES_CSV, nrows=0).columns)
missing_columns = expected_columns.difference(candidate_columns)
if missing_columns:
    raise ValueError(f"Missing candidate columns: {sorted(missing_columns)}")


def scan_candidate_counts(path: Path, chunk_size: int) -> Counter:
    counts = Counter()
    reader = pd.read_csv(path, usecols=["split", "label"], chunksize=chunk_size)
    for chunk in tqdm(reader, desc="Pass 1/2: counting RE candidates", unit="chunk"):
        grouped = chunk.groupby(["split", "label"]).size()
        counts.update({tuple(key): int(value) for key, value in grouped.items()})
    return counts


scan_started = time.perf_counter()
candidate_counts = scan_candidate_counts(RE_CANDIDATES_CSV, CANDIDATE_CHUNK_SIZE)
candidate_count_frame = (
    pd.Series(candidate_counts, name="count")
    .rename_axis(["split", "label"])
    .reset_index()
)
total_candidate_rows = int(candidate_count_frame["count"].sum())
expected_candidate_rows = int(prep_summary["candidate_pairs"])
if total_candidate_rows != expected_candidate_rows:
    raise ValueError(
        f"Candidate CSV has {total_candidate_rows:,} rows; prep_summary expects {expected_candidate_rows:,}"
    )

print(f"Candidate counting completed in {(time.perf_counter() - scan_started) / 60:.2f} minutes")
print(candidate_count_frame.pivot(index="split", columns="label", values="count").fillna(0).astype(int))
print({"candidate_rows": total_candidate_rows, "documents": len(documents)})


In [ ]:
def split_sampling_plan(
    counts: Counter,
    train_negative_ratio: int | None,
    eval_negative_ratio: int | None,
) -> dict[str, dict[str, int]]:
    plan = {}
    for split in ["train", "validation", "test"]:
        positive_count = sum(
            count for (candidate_split, label), count in counts.items()
            if candidate_split == split and label != "no_relation"
        )
        negative_count = counts[(split, "no_relation")]
        ratio = train_negative_ratio if split == "train" else eval_negative_ratio
        selected_negative_count = (
            negative_count if ratio is None else min(negative_count, positive_count * ratio)
        )
        plan[split] = {
            "available_positives": positive_count,
            "available_negatives": negative_count,
            "selected_negatives": selected_negative_count,
        }
    return plan


def load_sampled_candidate_splits(
    path: Path,
    counts: Counter,
    chunk_size: int,
    random_seed: int,
) -> tuple[dict[str, pd.DataFrame], dict]:
    plan = split_sampling_plan(
        counts,
        TRAIN_NEGATIVE_TO_POSITIVE_RATIO,
        EVAL_NEGATIVE_TO_POSITIVE_RATIO,
    )
    selected_parts: dict[str, list[pd.DataFrame]] = {
        "train": [],
        "validation": [],
        "test": [],
    }
    seen_negatives = Counter()
    selected_negatives = Counter()

    reader = pd.read_csv(path, chunksize=chunk_size)
    for chunk_index, chunk in enumerate(
        tqdm(reader, desc="Pass 2/2: loading sampled RE splits", unit="chunk")
    ):
        for split_index, split in enumerate(["train", "validation", "test"]):
            split_chunk = chunk[chunk["split"] == split]
            if split_chunk.empty:
                continue

            positives = split_chunk[split_chunk["label"] != "no_relation"]
            if not positives.empty:
                selected_parts[split].append(positives.copy())

            negatives = split_chunk[split_chunk["label"] == "no_relation"]
            if negatives.empty:
                continue

            seen_negatives[split] += len(negatives)
            total_available = plan[split]["available_negatives"]
            total_target = plan[split]["selected_negatives"]
            desired_cumulative = (
                total_target * seen_negatives[split] // total_available
                if total_available else 0
            )
            quota = desired_cumulative - selected_negatives[split]
            if quota <= 0:
                continue
            if quota == len(negatives):
                selected = negatives.copy()
            else:
                selected = negatives.sample(
                    n=quota,
                    random_state=random_seed + chunk_index * 10 + split_index,
                )
            selected_parts[split].append(selected)
            selected_negatives[split] += len(selected)

    frames = {}
    maximums = {
        "train": MAX_TRAIN_SAMPLES,
        "validation": MAX_EVAL_SAMPLES,
        "test": MAX_TEST_SAMPLES,
    }
    for split, parts in selected_parts.items():
        frame = pd.concat(parts, ignore_index=True)
        frame = frame.sample(frac=1, random_state=random_seed).reset_index(drop=True)
        if maximums[split] is not None:
            frame = frame.head(maximums[split]).copy()
        frames[split] = frame

    for split in plan:
        if selected_negatives[split] != plan[split]["selected_negatives"]:
            raise ValueError(
                f"{split}: selected {selected_negatives[split]:,} negatives; "
                f"expected {plan[split]['selected_negatives']:,}"
            )
    return frames, plan


load_started = time.perf_counter()
sampled_frames, candidate_sampling_plan = load_sampled_candidate_splits(
    RE_CANDIDATES_CSV,
    candidate_counts,
    CANDIDATE_CHUNK_SIZE,
    RANDOM_SEED,
)

for split, frame in sampled_frames.items():
    memory_mb = frame.memory_usage(index=True, deep=True).sum() / (1024 ** 2)
    print(f"\n{split}: {len(frame):,} rows, approximately {memory_mb:,.1f} MB in pandas")
    print(frame["label"].value_counts().to_string())

print(f"Candidate loading completed in {(time.perf_counter() - load_started) / 60:.2f} minutes")
print("Sampling plan:", json.dumps(candidate_sampling_plan, indent=2))


In [ ]:
def entity_marker_type(label: str) -> str:
    normalised = str(label).strip().lower()
    if normalised.startswith("anatomy"):
        return "ANAT"
    if normalised.startswith("observation"):
        return "OBS"
    raise ValueError(f"Unsupported RadGraph entity label for typed markers: {label}")


def marker_tokens(role: str, entity_label: str) -> tuple[str, str]:
    if MARKER_DESIGN == "generic":
        return f"[{role}]", f"[/{role}]"
    entity_type = entity_marker_type(entity_label)
    return f"[{role}_{entity_type}]", f"[/{role}_{entity_type}]"


def build_marked_tokens(
    tokens: list[str],
    head_start: int,
    head_end: int,
    tail_start: int,
    tail_end: int,
    head_label: str,
    tail_label: str,
    context_window: int,
) -> list[str]:
    left = max(0, min(head_start, tail_start) - context_window)
    right = min(len(tokens), max(head_end, tail_end) + context_window + 1)
    head_open, head_close = marker_tokens("HEAD", head_label)
    tail_open, tail_close = marker_tokens("TAIL", tail_label)
    marked_tokens: list[str] = []

    for index in range(left, right):
        if index == head_start:
            marked_tokens.append(head_open)
        if index == tail_start:
            marked_tokens.append(tail_open)

        marked_tokens.append(tokens[index])

        if index == head_end:
            marked_tokens.append(head_close)
        if index == tail_end:
            marked_tokens.append(tail_close)

    return marked_tokens


def candidate_row_to_example(row) -> dict:
    tokens = tokens_by_doc[row.doc_id]
    marked_tokens = build_marked_tokens(
        tokens=tokens,
        head_start=int(row.head_start),
        head_end=int(row.head_end),
        tail_start=int(row.tail_start),
        tail_end=int(row.tail_end),
        head_label=row.head_label,
        tail_label=row.tail_label,
        context_window=CONTEXT_WINDOW,
    )
    return {
        "tokens": marked_tokens,
        "labels": label_to_id[row.label],
    }


def frame_to_examples(frame: pd.DataFrame) -> list[dict]:
    return [candidate_row_to_example(row) for row in frame.itertuples(index=False)]


dataset = DatasetDict(
    {
        split: Dataset.from_list(frame_to_examples(frame))
        for split, frame in sampled_frames.items()
    }
)

print(dataset)
print({"relation_labels": label_names, "marker_design": MARKER_DESIGN})


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, use_fast=True)
tokenizer.add_special_tokens({"additional_special_tokens": SPECIAL_TOKENS})
marker_token_ids = set(tokenizer.convert_tokens_to_ids(SPECIAL_TOKENS))

# Opening and closing IDs are paired explicitly so type-aware markers are handled
# in the same way as generic markers.
marker_id_pairs = {}
for opening_token in [token for token in SPECIAL_TOKENS if not token.startswith("[/")]:
    closing_token = "[/" + opening_token[1:]
    marker_id_pairs[tokenizer.convert_tokens_to_ids(opening_token)] = (
        tokenizer.convert_tokens_to_ids(closing_token)
    )

compaction_stats = {
    "examples": 0,
    "compacted_examples": 0,
    "maximum_uncompacted_length": 0,
}


def marker_intervals(input_ids: list[int]) -> list[tuple[int, int]]:
    intervals = []
    for opening_id, closing_id in marker_id_pairs.items():
        if opening_id not in input_ids:
            continue
        opening_index = input_ids.index(opening_id)
        try:
            closing_index = input_ids.index(closing_id, opening_index + 1)
        except ValueError as error:
            raise ValueError("An RE entity marker has no matching closing marker.") from error
        intervals.append((opening_index, closing_index))

    intervals.sort()
    if len(intervals) != 2:
        raise ValueError(
            f"Expected two marked entity spans before compaction; found {len(intervals)}."
        )
    return intervals


def compact_pair_input(input_ids: list[int]) -> list[int]:
    """Keep both marked entities and their nearest WordPiece context."""
    compaction_stats["examples"] += 1
    compaction_stats["maximum_uncompacted_length"] = max(
        compaction_stats["maximum_uncompacted_length"],
        len(input_ids),
    )

    original_marker_count = sum(
        int(token_id in marker_token_ids) for token_id in input_ids
    )
    if original_marker_count != 4:
        raise ValueError(
            f"Expected four RE markers before truncation; found {original_marker_count}."
        )
    if len(input_ids) <= MAX_LENGTH:
        return input_ids

    compaction_stats["compacted_examples"] += 1
    body = input_ids[1:-1]
    intervals = marker_intervals(body)

    # Reserve [CLS], a possible internal [SEP], and the final [SEP].
    body_budget = MAX_LENGTH - 3
    required_indices = {
        index
        for start, end in intervals
        for index in range(start, end + 1)
    }
    if len(required_indices) > body_budget:
        raise ValueError(
            "The two marked entity spans alone exceed the RE maximum sequence length."
        )

    def distance_to_entity(index: int) -> int:
        distances = []
        for start, end in intervals:
            if index < start:
                distances.append(start - index)
            elif index > end:
                distances.append(index - end)
            else:
                distances.append(0)
        return min(distances)

    context_candidates = sorted(
        (
            (distance_to_entity(index), index)
            for index in range(len(body))
            if index not in required_indices
        ),
        key=lambda item: (item[0], item[1]),
    )
    selected_indices = set(required_indices)
    remaining_budget = body_budget - len(required_indices)
    selected_indices.update(
        index for _, index in context_candidates[:remaining_budget]
    )

    ordered_indices = sorted(selected_indices)
    segments: list[list[int]] = []
    segment = [body[ordered_indices[0]]]
    previous_index = ordered_indices[0]
    for index in ordered_indices[1:]:
        if index != previous_index + 1:
            segments.append(segment)
            segment = []
        segment.append(body[index])
        previous_index = index
    segments.append(segment)

    if len(segments) > 2:
        raise ValueError(
            f"Pair compaction unexpectedly produced {len(segments)} context fragments."
        )

    compacted_ids = [tokenizer.cls_token_id]
    for segment_index, segment_ids in enumerate(segments):
        if segment_index:
            compacted_ids.append(tokenizer.sep_token_id)
        compacted_ids.extend(segment_ids)
    compacted_ids.append(tokenizer.sep_token_id)

    if len(compacted_ids) > MAX_LENGTH:
        raise ValueError(
            f"Compacted RE input has {len(compacted_ids)} tokens; expected <= {MAX_LENGTH}."
        )
    compacted_marker_count = sum(
        int(token_id in marker_token_ids) for token_id in compacted_ids
    )
    if compacted_marker_count != 4:
        raise ValueError(
            "RE pair compaction removed an entity marker: "
            f"retained {compacted_marker_count}/4 markers."
        )
    return compacted_ids


def tokenize_batch(batch: dict) -> dict:
    # Tokenise without blind right-side truncation. Long pairs are compacted around
    # both marked entities so that neither relation endpoint can be removed.
    raw_tokenized = tokenizer(
        batch["tokens"],
        is_split_into_words=True,
        truncation=False,
    )

    compacted_input_ids = [
        compact_pair_input(list(input_ids))
        for input_ids in raw_tokenized["input_ids"]
    ]
    tokenized = {
        "input_ids": compacted_input_ids,
        "attention_mask": [
            [1] * len(input_ids) for input_ids in compacted_input_ids
        ],
        "labels": batch["labels"],
    }
    if "token_type_ids" in raw_tokenized:
        tokenized["token_type_ids"] = [
            [0] * len(input_ids) for input_ids in compacted_input_ids
        ]
    return tokenized


tokenized_dataset = dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc=f"Tokenising {RE_EXPERIMENT_NAME}",
    load_from_cache_file=False,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tokenized_dataset)
print("Pair compaction audit:", compaction_stats)


In [ ]:
def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)

    precision, recall, macro_f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )
    positive_macro_precision, positive_macro_recall, positive_macro_f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            labels=positive_label_ids,
            average="macro",
            zero_division=0,
        )
    )
    positive_micro_precision, positive_micro_recall, positive_micro_f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            labels=positive_label_ids,
            average="micro",
            zero_division=0,
        )
    )

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": macro_f1,
        "positive_macro_precision": positive_macro_precision,
        "positive_macro_recall": positive_macro_recall,
        "positive_macro_f1": positive_macro_f1,
        "positive_micro_precision": positive_micro_precision,
        "positive_micro_recall": positive_micro_recall,
        "positive_micro_f1": positive_micro_f1,
    }


train_label_ids = np.array(tokenized_dataset["train"]["labels"], dtype=int)
label_counts = np.bincount(train_label_ids, minlength=len(label_names))
class_weights = label_counts.sum() / np.maximum(label_counts, 1)
class_weights = class_weights / class_weights.mean()
if CLASS_WEIGHT_CAP is not None:
    class_weights = np.minimum(class_weights, CLASS_WEIGHT_CAP)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

print(
    pd.DataFrame(
        {
            "label": label_names,
            "train_count": label_counts,
            "class_weight": class_weights,
        }
    )
)


In [ ]:
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights: torch.Tensor | None = None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        if self.class_weights is None:
            loss_fct = torch.nn.CrossEntropyLoss()
        else:
            loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label_names),
    id2label=id_to_label,
    label2id=label_to_id,
)
model.resize_token_embeddings(len(tokenizer))

args_kwargs = {
    "output_dir": str(MODEL_DIR),
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,
    "num_train_epochs": NUM_EPOCHS,
    "weight_decay": WEIGHT_DECAY,
    "lr_scheduler_type": LR_SCHEDULER_TYPE,
    "warmup_ratio": WARMUP_RATIO,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "max_grad_norm": MAX_GRAD_NORM,
    "save_strategy": "epoch",
    "logging_strategy": "steps",
    "logging_steps": 50,
    "load_best_model_at_end": True,
    "metric_for_best_model": "positive_macro_f1",
    "greater_is_better": True,
    "report_to": [],
    "seed": RANDOM_SEED,
    "save_total_limit": 2,
    "fp16": torch.cuda.is_available(),
}

signature = inspect.signature(TrainingArguments.__init__)
if "eval_strategy" in signature.parameters:
    args_kwargs["eval_strategy"] = "epoch"
else:
    args_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**args_kwargs)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights_tensor if USE_CLASS_WEIGHTED_LOSS else None,
)

trainer.train()

In [ ]:
test_metrics = trainer.evaluate(tokenized_dataset["test"], metric_key_prefix="test")
trainer.save_model(str(MODEL_DIR / "best_model"))
tokenizer.save_pretrained(str(MODEL_DIR / "best_model"))

metrics_path = RESULTS_DIR / f"re_{RE_EXPERIMENT_NAME}_metrics.json"
metrics_path.write_text(json.dumps(test_metrics, indent=2), encoding="utf-8")

run_config = {
    "run_name": RUN_NAME,
    "re_experiment_name": RE_EXPERIMENT_NAME,
    "model_checkpoint": MODEL_CHECKPOINT,
    "marker_design": MARKER_DESIGN,
    "special_tokens": SPECIAL_TOKENS,
    "max_length": MAX_LENGTH,
    "context_window": CONTEXT_WINDOW,
    "long_pair_handling": "nearest_wordpiece_context_compaction",
    "long_pair_fragment_separator": tokenizer.sep_token,
    "pair_compaction_audit": compaction_stats,
    "learning_rate": LEARNING_RATE,
    "num_epochs": NUM_EPOCHS,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "weight_decay": WEIGHT_DECAY,
    "random_seed": RANDOM_SEED,
    "train_negative_to_positive_ratio": TRAIN_NEGATIVE_TO_POSITIVE_RATIO,
    "eval_negative_to_positive_ratio": EVAL_NEGATIVE_TO_POSITIVE_RATIO,
    "candidate_chunk_size": CANDIDATE_CHUNK_SIZE,
    "candidate_sampling_plan": candidate_sampling_plan,
    "use_class_weighted_loss": USE_CLASS_WEIGHTED_LOSS,
    "class_weight_cap": CLASS_WEIGHT_CAP,
    "sample_sizes": {split: len(frame) for split, frame in sampled_frames.items()},
    "label_names": label_names,
    "threshold_selection_split": "validation",
    "positive_threshold_grid": POSITIVE_THRESHOLD_GRID,
}
config_path = RESULTS_DIR / f"re_{RE_EXPERIMENT_NAME}_run_config.json"
config_path.write_text(json.dumps(run_config, indent=2), encoding="utf-8")

print(json.dumps(test_metrics, indent=2))
print("Saved metrics:", metrics_path)
print("Saved run config:", config_path)


In [ ]:
def predict_split(split: str) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    output = trainer.predict(tokenized_dataset[split])
    logits = output.predictions
    gold_ids = output.label_ids.astype(int)
    probabilities = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    frame = sampled_frames[split].reset_index(drop=True).copy()
    assert len(frame) == len(probabilities)
    return frame, gold_ids, probabilities


def threshold_predictions(probabilities: np.ndarray, threshold: float) -> np.ndarray:
    positive_probabilities = probabilities[:, positive_label_ids]
    best_positive_offsets = positive_probabilities.argmax(axis=1)
    best_positive_ids = np.array(positive_label_ids, dtype=int)[best_positive_offsets]
    best_positive_scores = positive_probabilities.max(axis=1)
    return np.where(
        best_positive_scores >= threshold,
        best_positive_ids,
        no_relation_id,
    )


def positive_metrics(gold_ids: np.ndarray, predicted_ids: np.ndarray) -> dict:
    micro_precision, micro_recall, micro_f1, _ = precision_recall_fscore_support(
        gold_ids,
        predicted_ids,
        labels=positive_label_ids,
        average="micro",
        zero_division=0,
    )
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        gold_ids,
        predicted_ids,
        labels=positive_label_ids,
        average="macro",
        zero_division=0,
    )
    return {
        "positive_micro_precision": micro_precision,
        "positive_micro_recall": micro_recall,
        "positive_micro_f1": micro_f1,
        "positive_macro_precision": macro_precision,
        "positive_macro_recall": macro_recall,
        "positive_macro_f1": macro_f1,
    }


validation_frame, validation_gold_ids, validation_probabilities = predict_split("validation")
threshold_rows = []
for threshold in POSITIVE_THRESHOLD_GRID:
    validation_predicted_ids = threshold_predictions(validation_probabilities, threshold)
    threshold_rows.append(
        {
            "threshold": float(threshold),
            **positive_metrics(validation_gold_ids, validation_predicted_ids),
            "predicted_positive_relations": int(
                np.isin(validation_predicted_ids, positive_label_ids).sum()
            ),
        }
    )

threshold_search = pd.DataFrame(threshold_rows).sort_values(
    ["positive_macro_f1", "positive_micro_f1", "positive_micro_precision"],
    ascending=False,
)
selected_threshold = float(threshold_search.iloc[0]["threshold"])
threshold_search_path = RESULTS_DIR / f"re_{RE_EXPERIMENT_NAME}_threshold_search.csv"
threshold_search.sort_values("threshold").to_csv(threshold_search_path, index=False)

threshold_config = {
    "re_experiment_name": RE_EXPERIMENT_NAME,
    "selection_split": "validation",
    "selection_metric": "positive_macro_f1",
    "selected_positive_threshold": selected_threshold,
    "random_seed": RANDOM_SEED,
    "best_validation_metrics": threshold_search.iloc[0].to_dict(),
}
threshold_config_path = RESULTS_DIR / f"re_{RE_EXPERIMENT_NAME}_selected_threshold.json"
threshold_config_path.write_text(json.dumps(threshold_config, indent=2), encoding="utf-8")

test_frame, test_gold_ids, test_probabilities = predict_split("test")
test_argmax_ids = test_probabilities.argmax(axis=1)
test_predicted_ids = threshold_predictions(test_probabilities, selected_threshold)
test_metrics_thresholded = positive_metrics(test_gold_ids, test_predicted_ids)
test_metrics_thresholded.update(
    {
        "selected_positive_threshold": selected_threshold,
        "accuracy": accuracy_score(test_gold_ids, test_predicted_ids),
        "gold_positive_relations": int(np.isin(test_gold_ids, positive_label_ids).sum()),
        "predicted_positive_relations": int(
            np.isin(test_predicted_ids, positive_label_ids).sum()
        ),
    }
)

test_frame["gold_label"] = [id_to_label[int(idx)] for idx in test_gold_ids]
test_frame["argmax_label"] = [id_to_label[int(idx)] for idx in test_argmax_ids]
test_frame["predicted_label"] = [id_to_label[int(idx)] for idx in test_predicted_ids]
test_frame["prediction_confidence"] = test_probabilities.max(axis=1)
test_frame["selected_positive_threshold"] = selected_threshold
test_frame["is_correct"] = test_frame["gold_label"] == test_frame["predicted_label"]
for label_id, label in id_to_label.items():
    test_frame[f"probability_{label}"] = test_probabilities[:, int(label_id)]

prediction_columns = [
    "source",
    "doc_id",
    "dataset",
    "split",
    "head_entity_id",
    "tail_entity_id",
    "head_start",
    "head_end",
    "tail_start",
    "tail_end",
    "head_label",
    "tail_label",
    "distance",
    "is_gold",
    "gold_label",
    "argmax_label",
    "predicted_label",
    "prediction_confidence",
    "selected_positive_threshold",
    "is_correct",
] + [f"probability_{label}" for label in label_names]

prediction_path = PREDICTIONS_DIR / f"re_{RE_EXPERIMENT_NAME}_test_predictions.csv"
test_frame[prediction_columns].to_csv(prediction_path, index=False)

thresholded_metrics_path = RESULTS_DIR / f"re_{RE_EXPERIMENT_NAME}_thresholded_metrics.json"
thresholded_metrics_path.write_text(
    json.dumps(test_metrics_thresholded, indent=2),
    encoding="utf-8",
)

report = classification_report(
    test_gold_ids,
    test_predicted_ids,
    labels=list(range(len(label_names))),
    target_names=label_names,
    digits=4,
    zero_division=0,
)
report_path = RESULTS_DIR / f"re_{RE_EXPERIMENT_NAME}_thresholded_classification_report.txt"
report_path.write_text(report, encoding="utf-8")

run_config["selected_positive_threshold"] = selected_threshold
run_config["thresholded_test_metrics"] = test_metrics_thresholded
config_path.write_text(json.dumps(run_config, indent=2), encoding="utf-8")

display(threshold_search.sort_values("threshold"))
print(json.dumps(test_metrics_thresholded, indent=2))
print(report)
print(
    "Saved:",
    threshold_search_path,
    threshold_config_path,
    prediction_path,
    thresholded_metrics_path,
    report_path,
    sep="\n  ",
)
